# 05 - Manual LangChain Agent Loop

## Learning objective

Expose the model-tool loop explicitly while retaining LangChain tools, messages, and normalized tool-call structures.

```text
Gemini -> AIMessage tool call -> Python tool -> ToolMessage -> Gemini -> final answer
```

This notebook breaks the implementation into small, inspectable steps.

> **Quota note:** One complete run usually makes three Gemini requests for the shopping example.


## 1. Check the active Python kernel

The executable should point to the Conda environment used for notebooks, such as `nlp_env`.


In [4]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

If the course packages are not installed in the active kernel, remove the `#` and run the cell once. Then use **Kernel -> Restart Kernel**.


In [5]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL and environment variables

`truststore` uses the Windows certificate store. The existing `.env` is loaded from the sibling script folder without printing secret values.


In [6]:
import os
from pathlib import Path

import truststore
from dotenv import load_dotenv

truststore.inject_into_ssl()

ENV_PATH = Path("../langchain-course/.env").resolve()
load_dotenv(dotenv_path=ENV_PATH)

print(".env exists:", ENV_PATH.exists())
print("Google key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))


.env exists: True
Google key loaded: True
Tavily key loaded: True
LangSmith key loaded: True
LangSmith tracing: true


## 4. Import LangChain tools and message types


In [7]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langsmith import traceable

MAX_ITERATIONS = 10
MODEL = "gemini-3.5-flash-lite" #"gemini-3.6-flash"


## 5. Define the two LangChain tools

`@tool` creates the name, description, argument schema, and executable LangChain tool.


In [8]:
@tool
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    prices = {
        "laptop": 1299.99,
        "headphones": 149.95,
        "keyboard": 89.50,
    }
    return prices.get(product, 0)


@tool
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.

    Available tiers: bronze, silver, gold.
    """
    discounts = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discounts.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)


## 6. Inspect and test tools without Gemini


In [9]:
tools = [get_product_price, apply_discount]
tools_dict = {current_tool.name: current_tool for current_tool in tools}

print("Tool names:", list(tools_dict))
print("Price schema:", get_product_price.args_schema.model_json_schema())
print("Discount schema:", apply_discount.args_schema.model_json_schema())


Tool names: ['get_product_price', 'apply_discount']
Price schema: {'description': 'Look up the price of a product in the catalog.', 'properties': {'product': {'title': 'Product', 'type': 'string'}}, 'required': ['product'], 'title': 'get_product_price', 'type': 'object'}
Discount schema: {'description': 'Apply a discount tier to a price and return the final price.\n\nAvailable tiers: bronze, silver, gold.', 'properties': {'price': {'title': 'Price', 'type': 'number'}, 'discount_tier': {'title': 'Discount Tier', 'type': 'string'}}, 'required': ['price', 'discount_tier'], 'title': 'apply_discount', 'type': 'object'}


In [10]:
import json

In [11]:
print(
    json.dumps(
        [get_product_price.args_schema.model_json_schema(), apply_discount.args_schema.model_json_schema()],
        indent=2)
)

[
  {
    "description": "Look up the price of a product in the catalog.",
    "properties": {
      "product": {
        "title": "Product",
        "type": "string"
      }
    },
    "required": [
      "product"
    ],
    "title": "get_product_price",
    "type": "object"
  },
  {
    "description": "Apply a discount tier to a price and return the final price.\n\nAvailable tiers: bronze, silver, gold.",
    "properties": {
      "price": {
        "title": "Price",
        "type": "number"
      },
      "discount_tier": {
        "title": "Discount Tier",
        "type": "string"
      }
    },
    "required": [
      "price",
      "discount_tier"
    ],
    "title": "apply_discount",
    "type": "object"
  }
]


In [7]:
price = get_product_price.invoke({"product": "laptop"})
final_price = apply_discount.invoke({"price": price, "discount_tier": "gold"})

print("Retrieved price:", price)
print("Discounted price:", final_price)


Retrieved price: 1299.99
Discounted price: 1000.99


## 7. Initialize Gemini and bind the tools

`bind_tools()` exposes the tools to the model. It does not execute them.


In [8]:
llm = init_chat_model(f"google_genai:{MODEL}")
llm_with_tools = llm.bind_tools(tools)

print(type(llm_with_tools))


<class 'langchain_core.language_models.chat_models._ChatModelBinding'>


## 8. Build the starting message history

The system message defines policy. The human message contains the question.


In [13]:
question = "What is the price of a laptop after applying a gold discount?"

messages = [
    SystemMessage(
        content=(
            "You are a helpful shopping assistant.\n"
            "You have access to a product catalog tool and a discount tool.\n\n "

            "STRICT RULES — you must follow these exactly:\n"
            
            "1. NEVER guess or assume any product price. "
            "You MUST call get_product_price first to get the real price.\n"
            
            "2. Only call apply_discount AFTER you have received "
            "a price from get_product_price. Pass the exact price "
            "returned by get_product_price — do NOT pass a made-up number.\n"
            
            "3. NEVER calculate discounts yourself using math. "
            "Always use the apply_discount tool.\n"
            
            "4. If the user does not specify a discount tier, "
            "ask them which tier to use — do NOT assume one."
        )
    ),
    HumanMessage(content=question),
]

for message in messages:
    print("\n##########################\n")
    print(type(message).__name__, "->", message.content)



##########################

SystemMessage -> You are a helpful shopping assistant.
You have access to a product catalog tool and a discount tool.

 STRICT RULES — you must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price — do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use — do NOT assume one.

##########################

HumanMessage -> What is the price of a laptop after applying a gold discount?


## 9. Optional one-step model inspection

> **This cell makes one Gemini request.** It is useful for inspecting the first normalized tool call before running the whole loop.


In [ ]:
# first_ai_message = llm_with_tools.invoke(messages)
# print("AI message type:", type(first_ai_message))
# print("Tool calls:", first_ai_message.tool_calls)


## 10. Define the complete manual loop

The loop stops when the model returns no tool call. It processes only the first tool call per iteration to make dependent actions easy to follow.


In [14]:
@traceable(name="LangChain Agent Loop")
def run_agent(question: str):
    messages = [
        SystemMessage(
            content=(
                
                "You are a helpful shopping assistant.\n"
                "You have access to a product catalog tool and a discount tool.\n\n "
    
                "STRICT RULES — you must follow these exactly:\n"
                
                "1. NEVER guess or assume any product price. "
                "You MUST call get_product_price first to get the real price.\n"
                
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            )
        ),
        HumanMessage(content=question),
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"--- Iteration {iteration} ---")

        ai_message = llm_with_tools.invoke(messages)
        tool_calls = ai_message.tool_calls

        if not tool_calls:
            return ai_message.text, messages + [ai_message]

        tool_call = tool_calls[0]
        tool_name = tool_call.get("name")
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call.get("id")

        print("Tool selected:", tool_name)
        print("Arguments:", tool_args)

        selected_tool = tools_dict.get(tool_name)
        if selected_tool is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = selected_tool.invoke(tool_args)
        print("Observation:", observation)

        messages.append(ai_message)
        messages.append(
            ToolMessage(
                content=str(observation),
                tool_call_id=tool_call_id,
            )
        )

    raise RuntimeError("Maximum iterations reached")


## 11. Run the manual agent

> **This cell normally makes three Gemini requests.**


In [15]:
final_answer, final_messages = run_agent(
    "What is the price of a laptop after applying a gold discount?"
)

print("Final answer:", final_answer)


--- Iteration 1 ---
Tool selected: get_product_price
Arguments: {'product': 'laptop'}
Observation: 1299.99
--- Iteration 2 ---
Tool selected: apply_discount
Arguments: {'price': 1299.99, 'discount_tier': 'gold'}
Observation: 1000.99
--- Iteration 3 ---
Final answer: The price of a laptop after applying a gold discount is $1,000.99.


## 12. Inspect message accumulation


In [16]:
for index, message in enumerate(final_messages):
    print("=" * 70)
    print(index, type(message).__name__)
    print("Tool calls:", getattr(message, "tool_calls", None))
    print("Content:", str(message.content)[:800])


0 SystemMessage
Tool calls: None
Content: You are a helpful shopping assistant.
You have access to a product catalog tool and a discount tool.

 STRICT RULES — you must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price — do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use — do NOT assume one.
1 HumanMessage
Tool calls: None
Content: What is the price of a laptop after applying a gold discount?
2 AIMessage
Tool calls: [{'name': 'get_product_price', 'args': {'product': 'laptop'}, 'id': 'call_87687', 'type': 'tool_call'}]
Content: []
3 ToolMessage
Tool calls: None
Content: 1299.99
4 AIMessage
Tool calls: [{'name': 'apply_discount', 'args': {'p

## 13. Key takeaways

- `bind_tools()` describes tools to the model.
- The model requests a tool but does not execute it.
- Python dispatches the requested tool.
- `ToolMessage` returns the real observation and links it to `tool_call_id`.
- The history grows after every model and tool step.
- `MAX_ITERATIONS` is a safety guard.

## Practice exercises

- Ask without specifying a discount tier.
- Change the product.
- Print the history length at each iteration.
- Compare this explicit loop with `create_agent()`.
